<a href="https://colab.research.google.com/github/Rishabh-Kasaudhan/DEMO/blob/main/Optuna_Basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 6.7 MB/s eta 0:00:00


In [15]:
# Import necessary libraries
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the Pima Indian Diabetes dataset from sklearn
# Note: Scikit-learn's built-in 'load_diabetes' is a regression dataset.
# We will load the actual diabetes dataset from an external source
import pandas as pd

# Load the Pima Indian Diabetes dataset (from UCI repository)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

# Load the dataset
df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [16]:
import numpy as np

# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# Impute the missing values with the mean of the respective column
df.fillna(df.mean(), inplace=True)

# Check if there are any remaining missing values
print(df.isnull().sum())


Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [17]:
# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into training and test sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional: Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')


Training set shape: (537, 8)
Test set shape: (231, 8)


In [18]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

def objective(trial):
  n_estimators=trial.suggest_int('n_estimators', 50, 200)
  max_depth=trial.suggest_int('max_depth', 3, 20)

  model=RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth,random_state=42)
  score=cross_val_score(model, X_train, y_train, cv=3,scoring='accuracy').mean()
  return score

In [19]:
study=optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler())
study.optimize(objective, n_trials=50)

[I 2026-06-13 12:57:53,982] A new study created in memory with name: no-name-f759241c-511e-4f85-a247-53e4d5810dec
[I 2026-06-13 12:57:54,567] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 96, 'max_depth': 12}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-06-13 12:57:55,304] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 121, 'max_depth': 8}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-06-13 12:57:55,890] Trial 2 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 105, 'max_depth': 4}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-06-13 12:57:56,544] Trial 3 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 104, 'max_depth': 10}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-06-13 12:57:57,639] Trial 4 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 182, 'max_depth': 15}. Best is trial 4 with value: 0.7709497

In [20]:
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7821229050279329
Best hyperparameters: {'n_estimators': 119, 'max_depth': 19}


In [22]:
from sklearn.metrics import accuracy_score
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Test accuracy: {accuracy}')

Test accuracy: 0.7445887445887446


In [23]:
# For visualizations
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [24]:
# 1. Optimization History
plot_optimization_history(study).show()

In [25]:
# 2. Parallel Coordinates Plot
plot_parallel_coordinate(study).show()

In [26]:
plot_param_importances(study).show()

Optimizing ML **Models** **bold text**

In [27]:
# Importing the required libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [39]:
# Define the objective function for Optuna
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [40]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2026-06-13 13:11:01,838] A new study created in memory with name: no-name-fd622336-9391-45fa-9743-69de416220d2
[I 2026-06-13 13:11:01,932] Trial 0 finished with value: 0.7579143389199255 and parameters: {'classifier': 'SVM', 'C': 13.312996887621152, 'kernel': 'rbf', 'gamma': 'auto'}. Best is trial 0 with value: 0.7579143389199255.
[I 2026-06-13 13:11:01,990] Trial 1 finished with value: 0.7206703910614524 and parameters: {'classifier': 'SVM', 'C': 0.2345333491734598, 'kernel': 'poly', 'gamma': 'auto'}. Best is trial 0 with value: 0.7579143389199255.
[I 2026-06-13 13:11:05,341] Trial 2 finished with value: 0.7355679702048418 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 299, 'learning_rate': 0.1448941528283569, 'max_depth': 6, 'min_samples_split': 10, 'min_samples_leaf': 9}. Best is trial 0 with value: 0.7579143389199255.
[I 2026-06-13 13:11:06,965] Trial 3 finished with value: 0.7486033519553073 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 

In [41]:
# Retrieve the best trial
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifier': 'SVM', 'C': 0.11600803708316502, 'kernel': 'linear', 'gamma': 'auto'}
Best trial accuracy: 0.7895716945996275
